# Gold-layer daily KPI table

This notebook creates a Gold-layer daily KPI table for banking analytics and reporting.

The purpose is to aggregate daily banking metrics from multiple Silver-layer tables and prepare centralized KPIs for dashboards and business monitoring.

The notebook calculates:

- daily transaction volume
- total transaction amount
- total customers
- total accounts
- total account balance
- average credit score
- number of high-risk customers

Final output:
banking.gold.daily_bank_kpi

In [0]:
%sql
CREATE OR REPLACE TABLE banking.gold.daily_bank_kpi AS

-- — Daily Transaction Aggregation
-- Metrics:
-- - total_transactions
-- - total_transaction_amount
-- DATE(txn_timestamp):
-- extracts only the transaction date.

WITH txn_daily AS (
    SELECT
        DATE(txn_timestamp) AS txn_date,
        COUNT(txn_id) AS total_transactions,
        SUM(amount) AS total_transaction_amount
    FROM banking.silver.transactions
    GROUP BY DATE(txn_timestamp)
),

-- — Customer Metrics

customer_metrics AS (
    SELECT
        COUNT(DISTINCT customer_id) AS total_customers
    FROM banking.silver.customers
),

-- — Account Metrics

-- Calculates:
-- - total accounts
-- - total balance across all accounts

account_metrics AS (
    SELECT
        COUNT(account_id) AS total_accounts,
        SUM(balance) AS total_balance
    FROM banking.silver.accounts
),

-- — Credit Risk Metrics
-- Calculates:
-- - average credit score
-- - number of high-risk customers

-- CASE statement counts customers
-- where risk_grade = 'HIGH'

credit_metrics AS (
    SELECT
        AVG(credit_score) AS avg_credit_score,
        SUM(
            CASE WHEN risk_grade='HIGH'
            THEN 1 ELSE 0 END
        ) AS high_risk_customers
    FROM banking.silver.credit_bureau_reports
)

--  — Build Final KPI Dataset
-- Combines all KPI metrics into one table.
-- CROSS JOIN is used because:
-- customer_metrics,
-- account_metrics,
-- and credit_metrics
-- each return only one row.

SELECT
t.txn_date,
cm.total_customers,
am.total_accounts,
am.total_balance,
t.total_transactions,
t.total_transaction_amount,
cr.avg_credit_score,
cr.high_risk_customers

FROM txn_daily t
CROSS JOIN customer_metrics cm
CROSS JOIN account_metrics am
CROSS JOIN credit_metrics cr

## Validate Table Creation

In [0]:
# Returns the row count as notebook output.
# Useful for:
# - workflows
# - orchestration pipelines

count = spark.sql("""
SELECT COUNT(*) AS cnt
FROM banking.gold.daily_bank_kpi
""").collect()[0]["cnt"]

dbutils.notebook.exit(str(count))